# Practice - full flow on gym_members.csv

A gym wants to know which members will renew their membership.

Do all the steps we did in class, on this new file. We want to predict `renewed`.

**Expected accuracy: about 0.82**

---
## Step 1 - Load and look

Expected shape: `(500, 10)`

In [1]:
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv("../Data/gym_members.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   member_id          500 non-null    str    
 1   age                500 non-null    float64
 2   plan_type          500 non-null    str    
 3   workouts_per_week  500 non-null    int64  
 4   session_minutes    442 non-null    float64
 5   distance_km        479 non-null    float64
 6   months_active      491 non-null    float64
 7   trainer            500 non-null    str    
 8   referral_source    500 non-null    str    
 9   renewed            500 non-null    str    
dtypes: float64(4), int64(1), str(5)
memory usage: 39.2 KB


In [5]:
df.describe().columns

Index(['age', 'workouts_per_week', 'session_minutes', 'distance_km',
       'months_active'],
      dtype='str')

---
## Step 2 - Separate numeric and categorical columns

Expected: 5 numeric, 3 categorical. `member_id` and `renewed` go in neither list.

In [6]:
nums_col = ['age', 'workouts_per_week', 'session_minutes', 'distance_km', 'months_active']
cat_col = ['plan_type', 'trainer', 'referral_source']

In [9]:
df.head(3)

,member_id,age,plan_type,workouts_per_week,session_minutes,distance_km,months_active,trainer,referral_source,renewed
0,GYM3230,43.0,Pro,3,56.0,9.3,35.0,No,Walk-in,No
1,GYM3391,16.0,Basic,3,NaN,10.6,17.0,No,Friend,No
2,GYM3137,16.0,Basic,4,61.0,5.2,2.0,No,Instagram,No


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   member_id          500 non-null    str    
 1   age                500 non-null    float64
 2   plan_type          500 non-null    str    
 3   workouts_per_week  500 non-null    int64  
 4   session_minutes    442 non-null    float64
 5   distance_km        479 non-null    float64
 6   months_active      491 non-null    float64
 7   trainer            500 non-null    str    
 8   referral_source    500 non-null    str    
 9   renewed            500 non-null    str    
dtypes: float64(4), int64(1), str(5)
memory usage: 39.2 KB


---
## Step 3 - Split into X and y

Expected: X `(500, 8)`, y `(500,)`

In [10]:
x = df.drop(columns = ['renewed', 'member_id'])
y = df['renewed']

---
## Step 4 - Handle missing values

Fill every numeric column with its own mean.

Expected before: `session_minutes` 58, `distance_km` 21, `months_active` 9. After: all zero.

In [14]:
df.isnull().sum()

member_id             0
age                   0
plan_type             0
workouts_per_week     0
session_minutes      58
distance_km          21
months_active         9
trainer               0
referral_source       0
renewed               0
dtype: int64

In [15]:
numeric_cols = ['session_minutes', 'distance_km', 'months_active']

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

---
## Step 5 - Encode and scale together

Use one `ColumnTransformer`.

Expected shape: `(500, 14)`

In [18]:
print(cat_col)

['plan_type', 'trainer', 'referral_source']


In [19]:
df['plan_type'].value_counts()

plan_type
Basic    241
Pro      164
Elite     95
Name: count, dtype: int64

In [20]:
df['trainer'].value_counts()

trainer
No     343
Yes    157
Name: count, dtype: int64

In [21]:
df['referral_source'].value_counts()

referral_source
Friend           166
Instagram        151
Walk-in          120
Advertisement     63
Name: count, dtype: int64

In [23]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()
encoded = encoder.fit_transform(x[cat_col])
print((encoded.shape))

(500, 9)


In [24]:
print(x.info())

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                500 non-null    float64
 1   plan_type          500 non-null    str    
 2   workouts_per_week  500 non-null    int64  
 3   session_minutes    442 non-null    float64
 4   distance_km        479 non-null    float64
 5   months_active      491 non-null    float64
 6   trainer            500 non-null    str    
 7   referral_source    500 non-null    str    
dtypes: float64(4), int64(1), str(3)
memory usage: 31.4 KB
None


In [25]:
print(type(encoded))

<class 'scipy.sparse._csr.csr_matrix'>


In [26]:
encoded_df = pd.DataFrame(encoded.toarray(), columns = encoder.get_feature_names_out(cat_col))
encoded_df.head()

,plan_type_Basic,plan_type_Elite,plan_type_Pro,trainer_No,trainer_Yes,referral_source_Advertisement,referral_source_Friend,referral_source_Instagram,referral_source_Walk-in
0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
1,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0


In [27]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaled_col=scaler.fit_transform(x[nums_col])
scaled_col.shape

(500, 5)

In [28]:
scaled_df = pd.DataFrame(scaled_col, columns=nums_col)
scaled_df.head()

,age,workouts_per_week,session_minutes,distance_km,months_active
0,2.267700,-0.019657,-0.393468,1.253649,1.106797
1,-1.908258,-0.019657,NaN,1.618643,-0.268800
2,-1.908258,0.635591,-0.107050,0.102517,-1.415131
3,-0.052277,1.290839,NaN,-0.655546,0.801109
4,0.566384,1.946088,NaN,-0.430935,-0.421644


In [29]:
scaled_df.describe().round(2)

,age,workouts_per_week,session_minutes,distance_km,months_active
count,500.00,500.00,442.00,479.00,491.00
mean,-0.00,0.00,0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00
min,-1.91,-1.99,-2.46,-1.25,-1.49
25%,-0.67,-0.67,-0.74,-0.71,-0.88
50%,0.03,-0.02,0.01,-0.23,-0.12
75%,0.72,0.64,0.69,0.44,0.80
max,2.89,1.95,3.27,5.66,2.10


In [30]:
from sklearn.compose import ColumnTransformer

In [32]:
preprocessor=ColumnTransformer([
    ('cat',OneHotEncoder(),cat_col),
    ('num',StandardScaler(),nums_col)
])

In [34]:
final_transform=preprocessor.fit_transform(x)
preprocessor.get_feature_names_out()

array(['cat__plan_type_Basic', 'cat__plan_type_Elite',
       'cat__plan_type_Pro', 'cat__trainer_No', 'cat__trainer_Yes',
       'cat__referral_source_Advertisement',
       'cat__referral_source_Friend', 'cat__referral_source_Instagram',
       'cat__referral_source_Walk-in', 'num__age',
       'num__workouts_per_week', 'num__session_minutes',
       'num__distance_km', 'num__months_active'], dtype=object)

In [35]:
final_transform.shape

(500, 14)

In [36]:
df.head(3)

,member_id,age,plan_type,workouts_per_week,session_minutes,distance_km,months_active,trainer,referral_source,renewed
0,GYM3230,43.0,Pro,3,56.000000,9.3,35.0,No,Walk-in,No
1,GYM3391,16.0,Basic,3,62.868778,10.6,17.0,No,Friend,No
2,GYM3137,16.0,Basic,4,61.000000,5.2,2.0,No,Instagram,No


In [37]:
final_df = pd.DataFrame(final_transform,
                        columns=preprocessor.get_feature_names_out())
final_df.head()

,cat__plan_type_Basic,cat__plan_type_Elite,cat__plan_type_Pro,cat__trainer_No,cat__trainer_Yes,cat__referral_source_Advertisement,cat__referral_source_Friend,cat__referral_source_Instagram,cat__referral_source_Walk-in,num__age,num__workouts_per_week,num__session_minutes,num__distance_km,num__months_active
0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,2.267700,-0.019657,-0.393468,1.253649,1.106797
1,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,-1.908258,-0.019657,NaN,1.618643,-0.268800
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,-1.908258,0.635591,-0.107050,0.102517,-1.415131
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.052277,1.290839,NaN,-0.655546,0.801109
4,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.566384,1.946088,NaN,-0.430935,-0.421644


---
## Step 6 - Train and test split

Expected: train `(400, 14)`, test `(100, 14)`

In [38]:
from sklearn.model_selection import train_test_split

In [39]:
x_train, x_test, y_train, y_test = train_test_split(
    final_transform, y, test_size=0.2, random_state=42)


print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(400, 14)
(100, 14)
(400,)
(100,)


---
## Step 7 - Train the model

`KNeighborsClassifier` with `n_neighbors=5`, then predict on the test set.

In [44]:
from sklearn.metrics import accuracy_score

In [45]:
accuracy_score(y_test, y_pred)

NameError: name 'y_pred' is not defined

In [40]:
from sklearn.neighbors import KNeighborsClassifier

In [42]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(x_train, y_train)

ValueError: Input X contains NaN.
KNeighborsClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [43]:
y_pred = model.predict(x_test)
y_pred[:10]

NotFittedError: This KNeighborsClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

---
## Step 8 - Score

Print the accuracy of your model.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
